In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_CRRI_Mathura_Road_Delhi_IMD_2023.xlsx")

In [4]:
df
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    32 non-null     float64
 2   February   26 non-null     float64
 3   March      24 non-null     float64
 4   April      26 non-null     float64
 5   May        34 non-null     float64
 6   June       28 non-null     float64
 7   July       27 non-null     float64
 8   August     27 non-null     float64
 9   September  17 non-null     float64
 10  October    19 non-null     float64
 11  November   25 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape


(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())


In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,290.000,115.000000,145.75,128.192308,69.000000,86.428571,87.851852,96.666667,75.588235,148.473684,327.00,369.000000
1,2,334.000,134.000000,145.75,106.000000,48.000000,61.000000,87.851852,96.666667,75.588235,148.473684,253.28,332.000000
2,3,300.000,145.000000,145.75,185.000000,104.000000,70.000000,87.851852,96.666667,75.588235,148.473684,253.28,277.000000
3,4,282.000,176.000000,145.75,113.000000,66.000000,125.000000,87.851852,82.000000,75.588235,148.473684,253.28,260.000000
4,5,289.000,196.000000,145.75,124.000000,127.000000,95.000000,97.000000,96.666667,75.588235,148.473684,253.28,228.000000
5,6,438.000,276.000000,145.75,149.000000,226.000000,77.000000,94.000000,96.666667,75.588235,148.473684,253.28,244.000000
6,7,400.000,246.000000,145.75,151.000000,130.000000,86.428571,93.000000,96.666667,75.588235,148.473684,407.00,269.857143
7,8,242.375,126.000000,145.75,189.000000,112.000000,101.000000,103.000000,96.000000,75.588235,148.473684,408.00,241.000000
8,9,242.375,200.000000,119.00,128.192308,175.000000,92.000000,100.000000,88.000000,75.588235,148.473684,253.28,266.000000
9,10,429.000,136.000000,145.75,128.192308,198.000000,75.000000,105.000000,116.000000,75.588235,148.473684,253.28,277.000000


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
